# HCFIM EEG Topographic Contrasts V1

This notebook generates MNE topographic projections comparing controls against Alzheimer's disease and frontotemporal dementia.

The figures are intended for the paper narrative. They do not replace the statistical rankings or classifier validation; they provide neurophysiological visualization of PSD and HCFIM regional/lateralized feature contrasts.


This notebook generates:

- PSD regional topomaps: control, disease and disease-control difference.
- Slowing-index regional topomaps: `(delta + theta) / (alpha + beta)`.
- HCFIM regional topomaps for selected geometry-ranked configurations.
- HCFIM lateralized_v1 topomaps for selected exploratory lateralized configurations.

Important interpretation note:

> Regional and lateralized feature values are projected back onto scalp electrodes for visualization only. These are not raw electrical-potential topographies.


## 1. Context and Methods

The subject-level feature matrices already contain regional and lateralized aggregations. To visualize them topographically, each regional value is assigned to the electrodes that define that region and then plotted using MNE's standard 10-20 scalp layout.

Main caveats:

- The topomap is a visual projection of derived features, not a voltage or power-density field directly measured at every electrode.
- Difference maps use `disease - control`.
- Cohen's d maps are included for effect-size orientation, but they are descriptive and should be interpreted together with the geometric ranking and bootstrap CIs.
- Temporal legacy labels are mapped for montage placement: `T3 -> T7`, `T4 -> T8`, `T5 -> P7`, `T6 -> P8`.


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import importlib.util
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.ticker import FormatStrFormatter

warnings.filterwarnings("ignore", category=UserWarning)

def resolve_project_root() -> Path:
    """Locate the repository without relying on a machine-specific path."""
    env_root = os.environ.get("HCFIM_PROJECT_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "notebooks").is_dir() and (candidate / "src").is_dir():
            return candidate
        raise FileNotFoundError(
            "HCFIM_PROJECT_ROOT does not point to the HCFIM-eeg-research repository."
        )

    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository. Run the notebook from within the repository "
        "or set HCFIM_PROJECT_ROOT."
    )

PROJECT_ROOT = resolve_project_root()
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RUN_GENERATE_TOPOFIGURES = True
SAVE_FIGURES = True
SHOW_FIGURES = True

RANDOM_STATE = 17536
MNE_IMPORT_TIMEOUT_SECONDS = 45
plt.rcParams["font.family"] = "Arial"

if importlib.util.find_spec("mne") is None:
    raise ImportError(
        "MNE is required for this notebook. Install it in the active kernel with `pip install mne` "
        "or run the notebook in the EEG analysis environment."
    )

check = subprocess.run(
    [sys.executable, "-B", "-c", "import mne; print(mne.__version__)"],
    capture_output=True,
    text=True,
    timeout=MNE_IMPORT_TIMEOUT_SECONDS,
)
if check.returncode != 0:
    raise RuntimeError(f"MNE import check failed: {check.stderr}")

import mne

print("Project root:", PROJECT_ROOT)
print("MNE version:", mne.__version__)

## 2. Cortical Schemas and Montage


In [ ]:
EEG_CHANNELS = [
    "Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8",
    "T3", "C3", "Cz", "C4", "T4",
    "T5", "P3", "Pz", "P4", "T6", "O1", "O2",
]

MONTAGE_ALIAS = {
    "T3": "T7",
    "T4": "T8",
    "T5": "P7",
    "T6": "P8",
}

REGIONAL_SCHEMA = {
    "frontal": ["Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8"],
    "temporal": ["T3", "T4", "T5", "T6"],
    "central": ["C3", "Cz", "C4"],
    "parietal": ["P3", "Pz", "P4"],
    "occipital": ["O1", "O2"],
}

LATERALIZED_SCHEMA = {
    "frontal_left": ["Fp1", "F7", "F3"],
    "frontal_right": ["Fp2", "F8", "F4"],
    "temporal_left": ["T3", "T5"],
    "temporal_right": ["T4", "T6"],
    "parietal_left": ["P3"],
    "parietal_right": ["P4"],
    "occipital_left": ["O1"],
    "occipital_right": ["O2"],
    "central_midline_extended": ["C3", "Cz", "C4", "Fz", "Pz"],
}


def build_topomap_positions(channels: list[str]) -> np.ndarray:
    montage = mne.channels.make_standard_montage("standard_1020")
    ch_pos = montage.get_positions()["ch_pos"]
    positions = []
    missing = []
    for ch in channels:
        montage_name = MONTAGE_ALIAS.get(ch, ch)
        if montage_name not in ch_pos:
            missing.append((ch, montage_name))
            positions.append([np.nan, np.nan])
        else:
            positions.append(ch_pos[montage_name][:2])
    if missing:
        raise ValueError(f"Missing montage coordinates: {missing}")
    return np.asarray(positions, dtype=float)


TOPO_POS = build_topomap_positions(EEG_CHANNELS)
print("Channels:", len(EEG_CHANNELS))
print("Regional groups:", list(REGIONAL_SCHEMA))
print("Lateralized groups:", list(LATERALIZED_SCHEMA))

## 3. Load Selected Subject-Level Features


In [ ]:
META_COLUMNS = ["participant_id", "diagnosis", "group", "age", "gender", "mmse", "n_epochs_aggregated"]

def hcfim_col(transform: str, distance: str, region: str, band: str, measure: str, agg: str = "mean") -> str:
    return f"hcfim__{transform}__{distance}__{region}__{band}__{measure}__{agg}"

def psd_col(region: str, band: str, agg: str = "mean") -> str:
    return f"psd__{region}__relative_bandpower_{band}__{agg}"

def read_selected_features(path: Path, selected_columns: list[str]) -> pd.DataFrame:
    header = pd.read_csv(path, nrows=0).columns.tolist()
    usecols = [c for c in META_COLUMNS if c in header] + selected_columns
    missing = sorted(set(selected_columns) - set(header))
    if missing:
        raise KeyError(f"{path.name} is missing selected columns: {missing[:10]} ... total={len(missing)}")
    return pd.read_csv(path, usecols=usecols)

PSD_BANDS = ["delta", "theta", "alpha", "beta"]
HCFIM_MEASURES = ["H", "C", "FIM"]
TRANSFORM = "ordinal_shannon_m4_t1"
DISTANCE = "euclidean"

regional_hcfim_requests = [
    {"comparison": "control_vs_alzheimer", "disease": "alzheimer", "band": "alpha", "schema": "regional_v1"},
    {"comparison": "control_vs_frontotemporal_dementia", "disease": "frontotemporal_dementia", "band": "broadband", "schema": "regional_v1"},
]

lateralized_hcfim_requests = [
    {"comparison": "control_vs_alzheimer", "disease": "alzheimer", "band": "theta", "schema": "lateralized_v1"},
    {"comparison": "control_vs_frontotemporal_dementia", "disease": "frontotemporal_dementia", "band": "broadband", "schema": "lateralized_v1"},
]

regional_selected = []
for region in REGIONAL_SCHEMA:
    for band in PSD_BANDS:
        regional_selected.append(psd_col(region, band))
    for request in regional_hcfim_requests:
        for measure in HCFIM_MEASURES:
            regional_selected.append(hcfim_col(TRANSFORM, DISTANCE, region, request["band"], measure))

lateralized_selected = []
for region in LATERALIZED_SCHEMA:
    for request in lateralized_hcfim_requests:
        for measure in HCFIM_MEASURES:
            lateralized_selected.append(hcfim_col(TRANSFORM, DISTANCE, region, request["band"], measure))

regional_df = read_selected_features(RESULTS_DIR / "subject_features_full.csv", sorted(set(regional_selected)))
lateralized_df = read_selected_features(RESULTS_DIR / "subject_features_lateralized_v1_full.csv", sorted(set(lateralized_selected)))

print("regional_df:", regional_df.shape)
print("lateralized_df:", lateralized_df.shape)
display(regional_df[["participant_id", "diagnosis"]].groupby("diagnosis").size().rename("n").reset_index())

## 4. Topomap Helper Functions


In [ ]:
def cohen_d(x: pd.Series, y: pd.Series) -> float:
    x = pd.Series(x).dropna().astype(float)
    y = pd.Series(y).dropna().astype(float)
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return np.nan
    pooled = np.sqrt(((nx - 1) * x.var(ddof=1) + (ny - 1) * y.var(ddof=1)) / (nx + ny - 2))
    if pooled == 0 or np.isnan(pooled):
        return 0.0
    return float((y.mean() - x.mean()) / pooled)

def regional_group_values(
    df: pd.DataFrame,
    schema: dict[str, list[str]],
    column_by_region,
    disease: str,
) -> pd.DataFrame:
    rows = []
    control_df = df.loc[df["diagnosis"] == "control"]
    disease_df = df.loc[df["diagnosis"] == disease]
    for region in schema:
        column = column_by_region(region)
        control_values = control_df[column]
        disease_values = disease_df[column]
        rows.append({
            "region": region,
            "column": column,
            "control_mean": control_values.mean(),
            "disease_mean": disease_values.mean(),
            "difference": disease_values.mean() - control_values.mean(),
            "cohen_d": cohen_d(control_values, disease_values),
            "n_control": control_values.notna().sum(),
            "n_disease": disease_values.notna().sum(),
        })
    return pd.DataFrame(rows)

def region_values_to_channels(region_table: pd.DataFrame, schema: dict[str, list[str]], value_column: str) -> np.ndarray:
    value_by_region = dict(zip(region_table["region"], region_table[value_column]))
    out = []
    for channel in EEG_CHANNELS:
        assigned = np.nan
        for region, region_channels in schema.items():
            if channel in region_channels:
                assigned = value_by_region.get(region, np.nan)
                break
        out.append(assigned)
    return np.asarray(out, dtype=float)

def plot_single_topomap(ax, values, title, *, cmap="viridis", vmin=None, vmax=None):
    kwargs = dict(
        axes=ax,
        show=False,
        contours=0,
        sensors=True,
        extrapolate="head",
    )
    try:
        im, _ = mne.viz.plot_topomap(values, TOPO_POS, cmap=cmap, vlim=(vmin, vmax), **kwargs)
    except TypeError:
        im, _ = mne.viz.plot_topomap(values, TOPO_POS, cmap=cmap, vmin=vmin, vmax=vmax, **kwargs)
    ax.set_title(title, fontsize=10)
    for label, (x, y) in zip(EEG_CHANNELS, TOPO_POS):
        ax.text(x, y, label, ha="center", va="center", fontsize=9, color="black")
    return im

def symmetric_limits(values: np.ndarray, minimum: float | None = None) -> tuple[float, float]:
    vmax = float(np.nanmax(np.abs(values)))
    if minimum is not None:
        vmax = max(vmax, minimum)
    if vmax == 0 or np.isnan(vmax):
        vmax = 1.0
    return -vmax, vmax

def save_or_show(fig, filename: str):
    output_path = FIGURES_DIR / filename
    if SAVE_FIGURES:
        fig.savefig(output_path, dpi=240, bbox_inches="tight")
        print("Saved:", output_path)
    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(fig)

## 5. PSD Regional Topomaps


In [ ]:
def plot_psd_band_contrast(comparison: str, disease: str):
    fig, axes = plt.subplots(len(PSD_BANDS), 3, figsize=(10.5, 12.0))
    fig.suptitle(f"PSD regional topographic projection | {comparison}", fontsize=14, y=0.995)

    for row_idx, band in enumerate(PSD_BANDS):
        table = regional_group_values(
            regional_df,
            REGIONAL_SCHEMA,
            lambda region, band=band: psd_col(region, band),
            disease,
        )
        control_values = region_values_to_channels(table, REGIONAL_SCHEMA, "control_mean")
        disease_values = region_values_to_channels(table, REGIONAL_SCHEMA, "disease_mean")
        diff_values = region_values_to_channels(table, REGIONAL_SCHEMA, "difference")

        shared_min = float(np.nanmin([control_values, disease_values]))
        shared_max = float(np.nanmax([control_values, disease_values]))
        diff_min, diff_max = symmetric_limits(diff_values)

        im0 = plot_single_topomap(axes[row_idx, 0], control_values, f"Control | {band}", cmap="viridis", vmin=shared_min, vmax=shared_max)
        im1 = plot_single_topomap(axes[row_idx, 1], disease_values, f"{disease} | {band}", cmap="viridis", vmin=shared_min, vmax=shared_max)
        im2 = plot_single_topomap(axes[row_idx, 2], diff_values, f"Difference | {band}", cmap="RdBu_r", vmin=diff_min, vmax=diff_max)

        fig.colorbar(im1, ax=axes[row_idx, :2], fraction=0.025, pad=0.02)
        fig.colorbar(im2, ax=axes[row_idx, 2], fraction=0.05, pad=0.04)

    fig.tight_layout(rect=[0, 0, 1, 0.98])
    save_or_show(fig, f"topomap_psd_regional_{comparison}.png")

In [ ]:
if RUN_GENERATE_TOPOFIGURES:
    plot_psd_band_contrast("control_vs_alzheimer", "alzheimer")
    plot_psd_band_contrast("control_vs_frontotemporal_dementia", "frontotemporal_dementia")

## 6. Slowing-Index Topomaps


In [ ]:
def compute_slowing_index_table(df: pd.DataFrame, schema: dict[str, list[str]], disease: str) -> pd.DataFrame:
    rows = []
    control_df = df.loc[df["diagnosis"] == "control"]
    disease_df = df.loc[df["diagnosis"] == disease]

    for region in schema:
        def si_values(group_df):
            delta = group_df[psd_col(region, "delta")]
            theta = group_df[psd_col(region, "theta")]
            alpha = group_df[psd_col(region, "alpha")]
            beta = group_df[psd_col(region, "beta")]
            return (delta + theta) / (alpha + beta)

        control_values = si_values(control_df)
        disease_values = si_values(disease_df)
        rows.append({
            "region": region,
            "control_mean": control_values.mean(),
            "disease_mean": disease_values.mean(),
            "difference": disease_values.mean() - control_values.mean(),
            "cohen_d": cohen_d(control_values, disease_values),
        })
    return pd.DataFrame(rows)


def plot_slowing_contrast(comparison: str, disease: str):
    table = compute_slowing_index_table(regional_df, REGIONAL_SCHEMA, disease)
    control_values = region_values_to_channels(table, REGIONAL_SCHEMA, "control_mean")
    disease_values = region_values_to_channels(table, REGIONAL_SCHEMA, "disease_mean")
    diff_values = region_values_to_channels(table, REGIONAL_SCHEMA, "difference")
    effect_values = region_values_to_channels(table, REGIONAL_SCHEMA, "cohen_d")

    shared_min = float(np.nanmin([control_values, disease_values]))
    shared_max = float(np.nanmax([control_values, disease_values]))
    diff_min, diff_max = symmetric_limits(diff_values)
    effect_min, effect_max = symmetric_limits(effect_values, minimum=0.2)

    fig, axes = plt.subplots(1, 4, figsize=(13.5, 3.8))
    fig.suptitle(f"Slowing index projection | {comparison}", fontsize=14, y=1.04)

    im0 = plot_single_topomap(axes[0], control_values, "Control", cmap="magma", vmin=shared_min, vmax=shared_max)
    im1 = plot_single_topomap(axes[1], disease_values, disease, cmap="magma", vmin=shared_min, vmax=shared_max)
    im2 = plot_single_topomap(axes[2], diff_values, "Disease - control", cmap="RdBu_r", vmin=diff_min, vmax=diff_max)
    im3 = plot_single_topomap(axes[3], effect_values, "Cohen's d", cmap="RdBu_r", vmin=effect_min, vmax=effect_max)

    fig.colorbar(im1, ax=axes[:2], fraction=0.025, pad=0.02)
    fig.colorbar(im2, ax=axes[2], fraction=0.045, pad=0.04)
    fig.colorbar(im3, ax=axes[3], fraction=0.045, pad=0.04)
    fig.tight_layout()
    save_or_show(fig, f"topomap_slowing_index_regional_{comparison}.png")
    display(table)


if RUN_GENERATE_TOPOFIGURES:
    plot_slowing_contrast("control_vs_alzheimer", "alzheimer")
    plot_slowing_contrast("control_vs_frontotemporal_dementia", "frontotemporal_dementia")

## 7. HCFIM Regional Topomaps


In [ ]:
def plot_hcfim_contrast(
    df: pd.DataFrame,
    schema: dict[str, list[str]],
    comparison: str,
    disease: str,
    band: str,
    schema_name: str,
    output_prefix: str,
):
    fig, axes = plt.subplots(len(HCFIM_MEASURES), 4, figsize=(13.5, 10.0))
    # fig.suptitle(
    #     f"HCFIM topographic projection | {schema_name} | {comparison} | {TRANSFORM}, {DISTANCE}, {band}",
    #     fontsize=13,
    #     y=0.995,
    # )
    plt.rcParams.update({'font.size': 12})
    plt.rcParams.update({
        'axes.titlesize': 10,     # Title size
        'axes.labelsize': 11,     # X and Y label size
        'xtick.labelsize': 10,    # X-tick size
        'ytick.labelsize': 10,    # Y-tick size
        'legend.fontsize': 10,    # Legend size
        'figure.titlesize': 12    # Super title size
    })

    summary_tables = []
    for row_idx, measure in enumerate(HCFIM_MEASURES):
        table = regional_group_values(
            df,
            schema,
            lambda region, measure=measure: hcfim_col(TRANSFORM, DISTANCE, region, band, measure),
            disease,
        )
        table["measure"] = measure
        summary_tables.append(table)

        control_values = region_values_to_channels(table, schema, "control_mean")
        disease_values = region_values_to_channels(table, schema, "disease_mean")
        diff_values = region_values_to_channels(table, schema, "difference")
        effect_values = region_values_to_channels(table, schema, "cohen_d")

        shared_min = float(np.nanmin([control_values, disease_values]))
        shared_max = float(np.nanmax([control_values, disease_values]))
        diff_min, diff_max = symmetric_limits(diff_values)
        effect_min, effect_max = symmetric_limits(effect_values, minimum=0.2)

        im0 = plot_single_topomap(axes[row_idx, 0], control_values, f"Control | {measure}", cmap="GnBu", vmin=shared_min, vmax=shared_max)
        im1 = plot_single_topomap(axes[row_idx, 1], disease_values, f"{disease} | {measure}", cmap="GnBu", vmin=shared_min, vmax=shared_max)
        im2 = plot_single_topomap(axes[row_idx, 2], diff_values, f"Difference | {measure}", cmap="coolwarm", vmin=diff_min, vmax=diff_max)
        im3 = plot_single_topomap(axes[row_idx, 3], effect_values, f"Cohen's d | {measure}", cmap="coolwarm", vmin=effect_min, vmax=effect_max)

        # fig.colorbar(im1, ax=axes[row_idx, :2], fraction=0.025, pad=0.02)
        # fig.colorbar(im2, ax=axes[row_idx, 2], fraction=0.045, pad=0.04)
        # fig.colorbar(im3, ax=axes[row_idx, 3], fraction=0.045, pad=0.04)

        cbar1 = fig.colorbar(im1, ax=axes[row_idx, :2], fraction=0.025, pad=0.02)
        cbar2 = fig.colorbar(im2, ax=axes[row_idx, 2], fraction=0.045, pad=0.04)
        cbar3 = fig.colorbar(im3, ax=axes[row_idx, 3], fraction=0.045, pad=0.04)
        
        cbar1.ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))
        cbar2.ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))
        cbar3.ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))
        
        cbar1.update_ticks()
        cbar2.update_ticks()
        cbar3.update_ticks()

    fig.tight_layout(rect=[0, 0, 1, 0.975])
    save_or_show(fig, f"{output_prefix}_{comparison}.png")
    return pd.concat(summary_tables, ignore_index=True)


hcfim_regional_summaries = []
if RUN_GENERATE_TOPOFIGURES:
    for request in regional_hcfim_requests:
        summary = plot_hcfim_contrast(
            regional_df,
            REGIONAL_SCHEMA,
            request["comparison"],
            request["disease"],
            request["band"],
            "regional_v1",
            "topomap_hcfim_regional",
        )
        summary["schema"] = "regional_v1"
        summary["comparison"] = request["comparison"]
        summary["band"] = request["band"]
        hcfim_regional_summaries.append(summary)

    hcfim_regional_summary = pd.concat(hcfim_regional_summaries, ignore_index=True)
    hcfim_regional_summary.to_csv(RESULTS_DIR / "topomap_hcfim_regional_summary.csv", index=False)
    display(hcfim_regional_summary.head(20))

## 8. HCFIM Lateralized_v1 Topomaps


In [ ]:
hcfim_lateralized_summaries = []
if RUN_GENERATE_TOPOFIGURES:
    for request in lateralized_hcfim_requests:
        summary = plot_hcfim_contrast(
            lateralized_df,
            LATERALIZED_SCHEMA,
            request["comparison"],
            request["disease"],
            request["band"],
            "lateralized_v1",
            "topomap_hcfim_lateralized_v1",
        )
        summary["schema"] = "lateralized_v1"
        summary["comparison"] = request["comparison"]
        summary["band"] = request["band"]
        hcfim_lateralized_summaries.append(summary)

    hcfim_lateralized_summary = pd.concat(hcfim_lateralized_summaries, ignore_index=True)
    hcfim_lateralized_summary.to_csv(RESULTS_DIR / "topomap_hcfim_lateralized_v1_summary.csv", index=False)
    display(hcfim_lateralized_summary.head(24))

## 9. Paper Caption Templates

In [ ]:
caption_templates = {
    "PSD": (
        "Topographic projection of regional PSD features. Each regional subject-level feature was assigned "
        "to the electrodes belonging to the corresponding cortical region and visualized on the standard 10-20 scalp layout. "
        "Difference maps show disease minus control."
    ),
    "Slowing index": (
        "Topographic projection of the slowing index, defined as (delta + theta) / (alpha + beta). "
        "Maps show group means, disease-control differences and Cohen's d effect-size projections."
    ),
    "HCFIM": (
        "Topographic projection of HCFIM regional/lateralized features for the selected geometry-ranked configuration. "
        "H, C and FIM are derived information-theoretic coordinates computed from ordinal Shannon probability distributions "
        "and projected back to scalp regions for visualization only."
    ),
}
for key, text in caption_templates.items():
    print(f"{key}:\n{text}\n")

## 10. Publication Figure 3

This final cell reproduces the regional broadband control-versus-FTD topographic figure used as Figure 3 in the revised manuscript. It uses the regional summary generated above, MNE's standard 10-20 montage and topographic interpolation, shared Control/FTD scales, zero-centered difference scales, a fixed Cohen's d scale of +/-1.1, and the 19 electrode labels. The interpolation visualizes five regional means and does not add electrode-level measurements.


In [ ]:
from hashlib import sha256

PUBLICATION_FIGURE_3 = FIGURES_DIR / "figure_3_regional_hcfim_ftd_mne_v6.png"


def publication_colorbar(fig, image, colorbar_axis, decimals):
    colorbar = fig.colorbar(image, cax=colorbar_axis)
    colorbar.ax.yaxis.set_major_formatter(FormatStrFormatter(f"%.{decimals}f"))
    colorbar.ax.tick_params(labelsize=8, width=0.7, length=3)
    colorbar.outline.set_linewidth(0.8)
    colorbar.update_ticks()


def publication_topomap(axis, values, title, cmap, limits):
    image, _ = mne.viz.plot_topomap(
        values,
        TOPO_POS,
        axes=axis,
        show=False,
        sensors=True,
        contours=0,
        outlines="head",
        extrapolate="head",
        border="mean",
        image_interp="cubic",
        res=256,
        cmap=cmap,
        vlim=limits,
    )
    axis.set_title(title, fontsize=11, fontweight="semibold", pad=9)
    for label, (x_coordinate, y_coordinate) in zip(EEG_CHANNELS, TOPO_POS):
        axis.text(
            x_coordinate,
            y_coordinate,
            label,
            ha="center",
            va="center",
            fontsize=7.2,
            color="#111111",
            zorder=5,
        )
    return image


def build_publication_figure_3(summary_table):
    figure_data = summary_table.loc[
        (summary_table["comparison"] == "control_vs_frontotemporal_dementia")
        & (summary_table["band"] == "broadband")
    ].copy()
    if len(figure_data) != 15:
        raise RuntimeError(f"Expected 15 regional rows, found {len(figure_data)}")

    plt.rcParams.update({
        "font.family": "Arial",
        "font.size": 9,
        "axes.titlecolor": "#111111",
        "figure.facecolor": "white",
    })
    figure = plt.figure(figsize=(15.2, 10.4), facecolor="white")
    grid = figure.add_gridspec(
        3,
        7,
        width_ratios=[1, 1, 0.045, 1, 0.045, 1, 0.045],
        wspace=0.22,
        hspace=0.30,
        left=0.035,
        right=0.975,
        top=0.895,
        bottom=0.065,
    )
    figure.suptitle(
        "",
        fontsize=18,
        fontweight="semibold",
        y=0.965,
    )
    figure.text(
        0.5,
        0.925,
        "",
        ha="center",
        va="center",
        fontsize=10.5,
        color="#555555",
    )

    for row_index, measure in enumerate(HCFIM_MEASURES):
        measure_table = figure_data.loc[figure_data["measure"] == measure].copy()
        control_values = region_values_to_channels(measure_table, REGIONAL_SCHEMA, "control_mean")
        disease_values = region_values_to_channels(measure_table, REGIONAL_SCHEMA, "disease_mean")
        difference_values = region_values_to_channels(measure_table, REGIONAL_SCHEMA, "difference")
        effect_values = region_values_to_channels(measure_table, REGIONAL_SCHEMA, "cohen_d")

        group_limits = (
            float(min(control_values.min(), disease_values.min())),
            float(max(control_values.max(), disease_values.max())),
        )
        difference_bound = float(np.max(np.abs(difference_values)))
        difference_limits = (-difference_bound, difference_bound)
        effect_limits = (-1.1, 1.1)

        control_axis = figure.add_subplot(grid[row_index, 0])
        disease_axis = figure.add_subplot(grid[row_index, 1])
        group_colorbar_axis = figure.add_subplot(grid[row_index, 2])
        difference_axis = figure.add_subplot(grid[row_index, 3])
        difference_colorbar_axis = figure.add_subplot(grid[row_index, 4])
        effect_axis = figure.add_subplot(grid[row_index, 5])
        effect_colorbar_axis = figure.add_subplot(grid[row_index, 6])

        #viridis
        #RdBu_r

        publication_topomap(control_axis, control_values, f"Control | {measure}", "GnBu", group_limits)
        group_image = publication_topomap(disease_axis, disease_values, f"FTD | {measure}", "GnBu", group_limits)
        difference_image = publication_topomap(
            difference_axis,
            difference_values,
            f"FTD − Control | {measure}",
            "RdBu_r",
            difference_limits,
        )
        effect_image = publication_topomap(
            effect_axis,
            effect_values,
            f"Cohen’s d | {measure}",
            "RdBu_r",
            effect_limits,
        )

        group_decimals = 3 if measure != "C" else 4
        publication_colorbar(figure, group_image, group_colorbar_axis, group_decimals)
        publication_colorbar(figure, difference_image, difference_colorbar_axis, 4 if measure == "C" else 3)
        publication_colorbar(figure, effect_image, effect_colorbar_axis, 1)

    figure.text(
        0.5,
        0.022,
        "",
        ha="center",
        va="center",
        fontsize=9,
        color="#555555",
    )
    figure.savefig(
        PUBLICATION_FIGURE_3,
        dpi=400,
        facecolor="white",
        bbox_inches="tight",
        pad_inches=0.10,
    )
    return figure


publication_figure_3 = build_publication_figure_3(hcfim_regional_summary)
publication_figure_3_hash = sha256(PUBLICATION_FIGURE_3.read_bytes()).hexdigest()
print("Saved:", PUBLICATION_FIGURE_3)
print("SHA256:", publication_figure_3_hash)
plt.show()
